## 15. 🆕 Modelo avanzado: sexo y edad de retiro, ρ con moratorias y migración

Hasta ahora $TBP_C$ trataba a la cohorte como un bloque único que se retira a los 65 con un $\rho$ y un $J$ únicos. Esta sección agrega tres refinamientos y los **compara paso a paso** con el $TBP$ base:

$$TBP_C(t)=\frac{N_t\,(1-\mu)\,\bar A\,\tau}{J\,\rho}\quad\text{(años; versión principal: }\mathbf{no\ contiene}\ E_r\text{)},\qquad TBP_A=\frac{TBP_C}{E_r},\qquad TBP_B=\frac{\bar A\,\tau}{E_r\,\rho}$$

1. **Sexo y edad de retiro (varones 65, mujeres 60).** El numerador se suma por sexo: $\sum_s N_s\,S_s(r_s)\,\bar A_s\,\tau$ con $N_s=N_t\cdot share_s$ (reparto por la razón de masculinidad al nacer $SRB$ de la ONU, no 51/49 fijo), $S_s(r_s)$ la supervivencia **cohortal** del sexo hasta *su* edad de retiro y $\bar A_s$ la suma de aportes de la EPH **del sexo** hasta $r_s-1$ (varones 18–64, mujeres 18–59). El denominador usa $J^*$ = varones 65+ **más** mujeres 60+: las mujeres de 60 a 64 ya cobran, así que son pasivas.
2. **$\rho$ con moratorias (ESCENARIO).** $\rho_{total}=s\,\rho_{mor}+(1-s)\,\rho_{sin\,mor}$, con $s$ = participación de beneficiarios con moratoria. Se despeja $\rho_{mor}$ implícito 2009–2023 y se plantean **tres supuestos** para cohortes que se retiran después de 2023 (no son predicciones).
3. **Migración.** En vez de "N + ΣM" (no calculable por cohorte) se usa la corrección empírica **implícita en la proyección ONU**: $m=\text{población de la edad de retiro}/(N\cdot S)$. Es opcional y **no es migración pura**.

> ⚠️ **Qué es y qué no es.** Todos los niveles son **condicionales a supuestos** (en especial $\bar A$, que es la cota inferior de asalariados de la EPH, y $\tau$ constante); no es una predicción y no hay ninguna "significancia" que reportar: son cuentas de un modelo determinista, no estimaciones con error muestral. Las diferencias entre pasos muestran **cuánto pesa cada supuesto**, no cuál es verdadero.

> **Errores de otro esquema que NO se copian:** (a) $TBP_C=N(r-\bar A)\tau/(J\rho E_r)$ rompe la definición (mete $E_r$ en $TBP_C$ y un término $(r-\bar A)$ inventado); (b) $E_r$ por sexo no se fabrica multiplicando por 0,92/1,08: se usan las tablas ONU por sexo; (c) "$N+\sum M_x$" no es calculable por cohorte; (d) "ponderar y luego sumar" mezcla dos agregaciones: acá se define una sola, explicada en 15.3.

Depende solo de `d`, `s`, `mort`, `TAU`, `RHO`, `A_OBS`, `A_LEGAL`, `DATA`, `OUT` en memoria, de los módulos de las secciones 12–14 (`tbp_eph`, `tbp_anses`, `tbp_mortalidad`, que **reusa**) y de archivos que carga con rutas candidatas (`../datos/procesados`, `datos/procesados`) y, si faltan, la URL *raw* del repositorio. **No modifica `d`.**

### 15.1 Módulo (copia de `src/tbp_modelo_avanzado.py`)

Todo el código está en una sola celda para que el Colab sea autosuficiente. El mismo archivo vive en el repositorio (`src/tbp_modelo_avanzado.py`; `python src/tbp_modelo_avanzado.py` corre el autotest con datos **sintéticos** rotulados). **Reusa** (no reescribe) `pipeline(..., sexo=)` de `tbp_eph`, `asignar_rho_tau_por_cohorte` de `tbp_anses` y `supervivencia_cohortal` / `er_cohortal` de `tbp_mortalidad`; si el notebook ya corrió las secciones 12–14, toma esas funciones de la memoria.

In [ ]:
# ====== MÓDULO tbp_modelo_avanzado (misma versión que src/tbp_modelo_avanzado.py) ======
# -*- coding: utf-8 -*-
"""
tbp_modelo_avanzado.py — TBP_C con sexo y edad de retiro, ρ con moratorias (escenarios) y ajuste por migración.

Proyecto TBP — Índice de Cuna Vacía (Taller de Indicadores).  Versión principal propuesta (años de aporte por año de jubilado):

        TBP_C(t) = N_t · (1−μ) · Ā · τ / (J · ρ)            [en el código: N_t · S65 · Ā · τ / (J · ρ)]
        TBP_A    = TBP_C / E_r          TBP_B = Ā·τ / (E_r·ρ)          (TBP_C NO contiene E_r)

QUÉ AGREGA ESTE MÓDULO (todo REUSA tbp_eph, tbp_anses y tbp_mortalidad; no los reescribe)
 1) SEXO Y EDAD DE RETIRO (principal: varones 65, mujeres 60 = edad legal en el SIPA).  Numerador aditivo por sexo:
        Σ_s  N_s · S_s(r_s) · Ā_s · τ ,     N_s = N_t · share_s ,  share_varones = SRB/(100+SRB)   (SRB ONU del año t)
    S_s(r_s) = supervivencia cohortal de ese sexo hasta su edad de retiro (tablas ONU por sexo); Ā_s = Σ_{a=18}^{r_s−1} S(a)·d_s(a)
    con la densidad d_s de la EPH del sexo s (varones 18-64; mujeres 18-59).  Denominador: J*·ρ con
        J* = varones 65+ + mujeres 60+ (ONU, año t+65): las mujeres de 60 a 64 ya cobran, por eso cuentan como pasivas.
    SENSIBILIDAD ("retiro a los 65 para todos", J = población 65+ total, modelo base): sobrecuenta aportes de las mujeres de 60 a 64
    (que ya no aportan) y omite del denominador a las pasivas de 60 a 64.
    AGREGACIÓN (definida, no "ponderada y luego sumada"): lo que se SUMA son las contribuciones al numerador
    (TBP_C_total = TBP_C_varones + TBP_C_mujeres, ambos sobre el MISMO J*·ρ).  Lo que se PROMEDIA (ponderando por los
    sobrevivientes al retiro N_s·S_s) son las intensidades por persona: Ā, E_r y TBP_B.  TBP_A total = Σ_s TBP_C_s / E_{r,s}
    y cumple TBP_A = D*·TBP_B con D* = Σ N_s S_s / J*.
 2) ρ CON MORATORIAS (ESCENARIO).  Como ρ = haber medio / salario imponible medio y el haber medio de todos los beneficiarios es
    el promedio ponderado por beneficiarios, ρ_total = s·ρ_mor + (1−s)·ρ_sin_mor  (s = participación de beneficiarios con moratoria).
    Con ρ_total, ρ_sin_mor y s de ANSES (digitalizados, aprox.) se DESPEJA ρ_mor implícito 2009-2023 y se verifica que sea plausible.
    Para cohortes que se retiran después de 2023, s es un SUPUESTO (no una predicción): 'persiste' (≈62,9 %), 'mitad' (≈31 %),
    'sin_moratorias' (0 %).  Mayor ρ => menor TBP_C.
 3) MIGRACIÓN.  En vez de "N + ΣM" (no calculable por cohorte) se usa la corrección empírica implícita en la proyección ONU:
        m = población de la edad de retiro en el año calendario del retiro / (N_s · S_s)      (N_efectivo · S = población ONU)
    Mezcla migración neta, diferencias entre fuentes (DEIS vs ONU) y el desfase de nacimientos: NO es migración pura.
    Opción `ajuste_migracion` True/False.

SUPUESTOS Y LÍMITES (rotulados en cada función): ρ y τ iguales para varones y mujeres (no hay series por sexo); ρ y τ del año t+65
    para ambos sexos; J* es un stock en t+65 (las mujeres se retiran 5 años antes); τ constante (supuesto); todos los niveles dependen de Ā;
    la EPH es urbana, solo asalariados (cota inferior).  Los resultados son condicionales a supuestos, NO son una predicción.

Uso mínimo:
    import tbp_modelo_avanzado as ma
    ma.autotest_modelo_avanzado()            # datos SINTÉTICOS rotulados
"""
import os
import sys
import importlib
import urllib.request

import numpy as np
import pandas as pd

RAW_BASE_MA = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
RUTAS_MA = ["../datos/procesados", "datos/procesados"]
RUTAS_SRC_MA = ["../src", "src", "github_staging/src"]
EDAD_V, EDAD_M = 65, 60                     # edades de retiro PRINCIPALES: varones 65, mujeres 60 (edad legal en el SIPA)
EDAD_ALT = 65                               # SENSIBILIDAD: "retiro a los 65 para todos" (modelo base) con J = población 65+ total
TAU_DEF, RHO_DEF, A_OBS_DEF, A_LEGAL_DEF = 0.2177, 0.403, 14.2, 30.0
ESCENARIOS_MOR = {"persiste": 1.0, "mitad": 0.5, "sin_moratorias": 0.0}   # multiplicador de la participación observada en 2023
ETQ_MOR = {"persiste": "persiste (≈62,9 %)", "mitad": "mitad (≈31 %)", "sin_moratorias": "sin moratorias (0 %)"}
ESC_NAT = ("persistencia", "recuperacion_lenta", "caida_adicional")
NOTA_ABAR = ("Ā por sexo: EPH (solo asalariados con descuento jubilatorio = cota inferior), pseudo-panel cohorte×edad, "
             "truncado a la edad de retiro del sexo (varones 18-64, mujeres 18-59).")


# ================================================================== 0) DEPENDENCIAS (reuso de los módulos del proyecto)
_NOMBRES_DEP = {"tbp_eph": ["pipeline", "supervivencia_gompertz", "EDADES", "find_eph_files", "read_all"],
                "tbp_anses": ["asignar_rho_tau_por_cohorte", "serie_rho_digitalizada", "serie_beneficiarios_digitalizada", "serie_tau"],
                "tbp_mortalidad": ["cargar_tablas", "supervivencia_cohortal", "er_cohortal"]}


class _Dep:
    pass


def dependencias():
    """Devuelve un objeto con las funciones reusadas de tbp_eph, tbp_anses y tbp_mortalidad.
    Orden: (1) nombres ya definidos en el espacio global (el notebook embebe los módulos de las secciones 12-14);
    (2) import desde src/ (rutas candidatas). Si falta alguno, error explícito."""
    g = globals()
    dep = _Dep()
    for mod, nombres in _NOMBRES_DEP.items():
        m = None
        if any(n not in g for n in nombres):
            for r in RUTAS_SRC_MA:
                if os.path.isdir(r) and os.path.abspath(r) not in sys.path:
                    sys.path.append(os.path.abspath(r))
            try:
                m = importlib.import_module(mod)
            except ImportError:
                m = None
        for n in nombres:
            if n in g:
                setattr(dep, n, g[n])
            elif m is not None and hasattr(m, n):
                setattr(dep, n, getattr(m, n))
            else:
                raise ImportError(f"falta `{n}` de {mod}: correr antes las secciones 12-14 del notebook o tener src/{mod}.py disponible")
    return dep


# ================================================================== 1) CARGA DE DATOS
def leer_csv_ma(nombre, rutas=None, destino="."):
    """Lee un CSV de datos/procesados: rutas candidatas ('../datos/procesados', 'datos/procesados' + las dadas);
    si no está, lo baja de GitHub (URL raw)."""
    for c in list(rutas or []) + RUTAS_MA:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return pd.read_csv(p, low_memory=False)
    os.makedirs(destino, exist_ok=True)
    p = os.path.join(destino, nombre)
    urllib.request.urlretrieve(f"{RAW_BASE_MA}/{nombre}", p)
    return pd.read_csv(p, low_memory=False)


def cargar_poblacion_sexo(df=None, rutas=None):
    """Población ONU por edad simple y SEXO (personas). Devuelve (PM, PF): DataFrames año × edad (0..100; 100 = grupo abierto 100+)."""
    if df is None:
        df = leer_csv_ma("un_wpp2024_argentina_poblacion_edad_simple.csv", rutas)
    if "Variant" in df.columns:
        df = df[df["Variant"] == "Medium"]
    out = []
    for col in ("PopMale", "PopFemale"):
        P = df.pivot_table(index="Time", columns="AgeGrpStart", values=col) * 1000.0
        P.columns = P.columns.astype(int)
        out.append(P)
    return out[0], out[1]


def cargar_srb(df=None, rutas=None):
    """SRB (varones por 100 mujeres al nacer, ONU) por año. Verifica que exista y sea razonable."""
    if df is None:
        df = leer_csv_ma("un_wpp2024_argentina_indicadores.csv", rutas)
    if "SRB" not in df.columns:
        raise ValueError("el CSV de indicadores ONU no tiene la columna SRB (razón de masculinidad al nacer)")
    if "Variant" in df.columns:
        df = df[df["Variant"] == "Medium"]
    srb = df.set_index("Time")["SRB"].astype(float)
    srb = srb.loc[1950:2100]                      # 2101 no trae SRB en la tabla ONU
    assert srb.notna().all() and srb.between(100, 112).all(), "SRB fuera del rango esperado (100-112 varones por 100 mujeres)"
    return srb


def cargar_natalidad_escenarios(df=None, rutas=None):
    """Escenarios de natalidad de la sección 9: DataFrame año × escenario (nacimientos, personas)."""
    if df is None:
        df = leer_csv_ma("natalidad_escenarios_2025_2035.csv", rutas)
    return df.pivot(index="anio", columns="escenario", values="nacimientos")


def share_varones(srb):
    """Participación de varones al nacer: SRB/(100+SRB).  SRB = varones por 100 mujeres (no se usa 51/49 fijo)."""
    return srb / (100.0 + srb)


# ================================================================== 2) SUPERVIVENCIA, E_r Y Ā POR SEXO
def supervivencia_retiro(cohortes, tablas, edad_v=EDAD_V, edad_m=EDAD_M):
    """S_s(r_s) por sexo: supervivencia COHORTAL (tablas ONU por sexo, trayectoria calendario) hasta la edad de retiro.
    Devuelve DataFrame: S_v (varones a edad_v), S_m (mujeres a edad_m), S_m65 (mujeres a 65),
    S_tot65 (Total a 65, = columna S65 de `d`)."""
    D = dependencias()
    co = [int(c) for c in cohortes]
    sv = D.supervivencia_cohortal(co, "Male", tablas)
    sf = D.supervivencia_cohortal(co, "Female", tablas)
    st = D.supervivencia_cohortal(co, "Total", tablas)
    return pd.DataFrame({"S_v": sv[edad_v], "S_m": sf[edad_m], "S_m65": sf[65], "S_tot65": st[65]}, index=pd.Index(co, name="cohorte"))


def er_retiro(cohortes, tablas, edad_v=EDAD_V, edad_m=EDAD_M):
    """E_r cohortal a la edad de retiro de cada sexo (varones a 65, mujeres a `edad_m`, 60 por defecto). Usa tbp_mortalidad.er_cohortal(edad=...)."""
    D = dependencias()
    co = [int(c) for c in cohortes]
    return pd.DataFrame({"E_v": D.er_cohortal(co, edad_v, "Male", tablas), "E_m": D.er_cohortal(co, edad_m, "Female", tablas)})


def abar_hasta_edad(dens, s65, edad_retiro):
    """Ā truncado: Σ_{a=18}^{edad_retiro−1} S_c(a)·d_c(a), con S de Gompertz calibrada a S(65)=s65 (igual que tbp_eph.abar_por_cohorte,
    que suma hasta 64).  Con edad_retiro=65 reproduce abar_por_cohorte; con 60 suma las edades 18-59."""
    D = dependencias()
    S = D.supervivencia_gompertz(s65.reindex(dens.index).values)
    cols = np.asarray(D.EDADES) < int(edad_retiro)
    return pd.Series((S * dens.values)[:, cols].sum(axis=1), index=dens.index)


def abar_por_sexo(df_eph, tablas, cohortes, edad_v=EDAD_V, edad_m=EDAD_M, ind_rule="solo_asalariados"):
    """Ā por sexo con la EPH REAL: corre tbp_eph.pipeline(..., sexo=1|2) (1 = varón, 2 = mujer) y trunca la suma de edades a r_s−1.
    Columnas: A_varones_{edad_v}, A_mujeres_{edad_m} (60 => 18-59) y A_mujeres_65 (sensibilidad) + n_edades_obs.
    Si no hay microdatos NO se inventa nada: quien llama debe omitir la parte por sexo."""
    D = dependencias()
    co = [int(c) for c in cohortes]
    out = {}
    for nombre, cod, sx, r_s in (("varones", 1, "Male", edad_v), ("mujeres", 2, "Female", edad_m)):
        s65 = D.supervivencia_cohortal(co, sx, tablas)[65]
        res = D.pipeline(df_eph, s65, ind_rule=ind_rule, sexo=cod, cohortes=co)
        for r in sorted({65, r_s}):
            out[f"A_{nombre}_{r}"] = abar_hasta_edad(res["dens"], s65, r)
        out[f"n_edades_obs_{nombre}"] = res["tabla"]["n_edades_obs"]
        # control interno: con r=65 la suma debe reproducir A_bar del pipeline
        assert np.allclose(out[f"A_{nombre}_65"].values, res["tabla"]["A_bar"].values, atol=1e-9), "Ā truncado a 65 != A_bar del pipeline"
    res = pd.DataFrame(out)
    res.index.name = "cohorte"
    return res


# ================================================================== 3) J* (PASIVOS POTENCIALES) Y REPONDERACIÓN
def pesos_cohorte_ma(s, nat, escenario):
    """w(c) = N_propio(c)/N_ONU(c) (misma regla que la sección 10): DEIS para 1950-2024 (w=1 donde DEIS tiene hueco), escenario de la sección 9
    para 2025-2035, w=1 en el resto.  Sirve para reponderar la población ONU por cohorte."""
    n_onu = s["un_births_miles"] * 1000.0
    n_deis = s["deis_nacimientos"]
    w = pd.Series(1.0, index=np.arange(1900, 2101))
    for c in range(1950, 2025):
        if not np.isnan(n_deis.get(c, np.nan)):
            w[c] = n_deis[c] / n_onu[c]
    for c in range(2025, 2036):
        w[c] = nat.loc[c, escenario] / n_onu[c]
    assert w.notna().all()
    return w


def j_pasivos(PM, PF, anios, edad_v=EDAD_V, edad_m=EDAD_M, w=None):
    """Pasivos potenciales en el año calendario y:  J* = varones edad_v+ + mujeres edad_m+  (ONU, edad simple, grupo 100+ incluido);
    J65 = población 65+ total (definición del modelo base).  `w` (Serie por cohorte, opcional) repondera como la sección 10:
    la población de edad a en el año y pertenece a la cohorte y−a (grupo abierto 100+ -> y−100).  Devuelve DataFrame por año."""
    filas = {}
    for y in anios:
        y = int(y)
        a_v = np.arange(edad_v, 101); a_m = np.arange(edad_m, 101); a_65 = np.arange(65, 101)
        wv = np.ones(len(a_v)) if w is None else w.reindex(y - a_v).values
        wm = np.ones(len(a_m)) if w is None else w.reindex(y - a_m).values
        w65 = np.ones(len(a_65)) if w is None else w.reindex(y - a_65).values
        jv = float((PM.loc[y, a_v].values * wv).sum())
        jm = float((PF.loc[y, a_m].values * wm).sum())
        j65 = float(((PM.loc[y, a_65].values + PF.loc[y, a_65].values) * w65).sum())
        filas[y] = dict(J_star=jv + jm, J_v=jv, J_m=jm, J65=j65)
    return pd.DataFrame.from_dict(filas, orient="index")


# ================================================================== 4) ρ CON MORATORIAS (ESCENARIO)
def rho_implicito(rho_total, rho_sin_mor, share_mor):
    """ρ_mor implícito:  ρ_total = s·ρ_mor + (1−s)·ρ_sin_mor  =>  ρ_mor = (ρ_total − (1−s)·ρ_sin_mor)/s.
    Es exacto si ambos ρ comparten el denominador (salario imponible medio) y la media del haber se pondera por beneficiarios;
    con datos digitalizados del gráfico es una APROXIMACIÓN."""
    s = np.asarray(share_mor, float)
    return (np.asarray(rho_total, float) - (1.0 - s) * np.asarray(rho_sin_mor, float)) / s


def tabla_rho_moratoria(serie_rho, ben):
    """Tabla anual 2009-2023: ρ_total, ρ_sin_mor, share_mor (ANSES, digitalizados) y ρ_mor implícito. Verifica 0<ρ_mor<1."""
    r = serie_rho.set_index("anio")[["rho", "rho_sin_moratoria"]]
    b = ben.set_index("anio")["share_moratoria"]
    t = r.join(b, how="inner")
    t["rho_mor_implicito"] = rho_implicito(t["rho"], t["rho_sin_moratoria"], t["share_moratoria"])
    assert t["rho_mor_implicito"].between(0, 1).all(), "ρ_mor implícito fuera de (0,1)"
    return t


def rho_por_escenario(anios, tabla_mor, escenario):
    """ρ_total por año calendario de retiro.  Años con dato (≤2023): ρ digitalizado (igual en todos los escenarios).
    Años > 2023: ρ = s_esc·ρ_mor(2023) + (1−s_esc)·ρ_sin_mor(2023), con s_esc = mult·s(2023) y mult ∈ {1 (persiste), 0,5 (mitad), 0 (sin moratorias)}.
    Los valores post-2023 son SUPUESTOS rotulados (no predicciones)."""
    if escenario not in ESCENARIOS_MOR:
        raise ValueError(f"escenario {escenario!r} no está en {list(ESCENARIOS_MOR)}")
    u = tabla_mor.index.max()
    s_u, rm_u, rsm_u = tabla_mor.loc[u, "share_moratoria"], tabla_mor.loc[u, "rho_mor_implicito"], tabla_mor.loc[u, "rho_sin_moratoria"]
    s_esc = ESCENARIOS_MOR[escenario] * s_u
    rho_post = s_esc * rm_u + (1.0 - s_esc) * rsm_u
    vals, tipo = [], []
    for y in anios:
        y = int(y)
        if y in tabla_mor.index:
            vals.append(float(tabla_mor.loc[y, "rho"])); tipo.append("dato (digitalizado)")
        elif y > u:
            vals.append(float(rho_post)); tipo.append(f"supuesto:{escenario}")
        else:
            vals.append(float(tabla_mor.loc[tabla_mor.index.min(), "rho"])); tipo.append("extrapolado:primer_valor")
    return pd.Series(vals, index=pd.Index([int(a) for a in anios], name="anio"), name=f"rho_{escenario}"), tipo


# ================================================================== 5) TBP_C POR SEXO, TBP_A, TBP_B
def tbp_c_sexos(N, sh_v, S_v, S_m, A_v, A_m, tau, rho, J, mig_v=1.0, mig_m=1.0):
    """TBP_C con numerador aditivo por sexo y denominador común J·ρ:
        TBP_C_s = N_t·share_s · S_s(r_s) · Ā_s · m_s · τ / (J·ρ)         TBP_C_total = TBP_C_varones + TBP_C_mujeres.
    Todas las entradas son Series por cohorte (o escalares).  `J` es J* (varones 65+ y mujeres 60+) o J65 (variante base).
    m_s = factor de migración (1 = sin ajuste).  Devuelve DataFrame con N_v, N_m, sobrevivientes al retiro y TBP_C."""
    N_v = N * sh_v
    N_m = N - N_v
    sob_v = N_v * S_v * mig_v
    sob_m = N_m * S_m * mig_m
    den = J * rho
    out = pd.DataFrame({"N_v": N_v, "N_m": N_m, "sob_v": sob_v, "sob_m": sob_m})
    out["TBP_C_varones"] = sob_v * A_v * tau / den
    out["TBP_C_mujeres"] = sob_m * A_m * tau / den
    out["TBP_C_total"] = out["TBP_C_varones"] + out["TBP_C_mujeres"]
    return out


def tbp_ab_sexos(parts, A_v, A_m, E_v, E_m, tau, rho, J):
    """TBP_A y TBP_B por sexo con E_r cohortal del sexo a su edad de retiro.
    TBP_A_s = TBP_C_s/E_s (aditivo: TBP_A_total = suma);  TBP_B_s = Ā_s·τ/(E_s·ρ) (por persona) y TBP_B_total = promedio ponderado por
    los sobrevivientes al retiro (N_s·S_s).  Identidad:  TBP_A_total = D*·TBP_B_total con D* = Σ sobrevivientes / J."""
    out = pd.DataFrame(index=parts.index)
    out["TBP_A_varones"] = parts["TBP_C_varones"] / E_v
    out["TBP_A_mujeres"] = parts["TBP_C_mujeres"] / E_m
    out["TBP_A_total"] = out["TBP_A_varones"] + out["TBP_A_mujeres"]
    out["TBP_B_varones"] = A_v * tau / (E_v * rho)
    out["TBP_B_mujeres"] = A_m * tau / (E_m * rho)
    sob = parts["sob_v"] + parts["sob_m"]
    out["TBP_B_total"] = (parts["sob_v"] * out["TBP_B_varones"] + parts["sob_m"] * out["TBP_B_mujeres"]) / sob
    out["D_star"] = sob / J
    out["E_r_implicito_total"] = parts["TBP_C_total"] / out["TBP_A_total"]
    return out


def descomponer_retiro_mujeres(N, sh_v, S_v, S_m65, S_m60, A_v, A_m65, A_m60, tau, rho, J65, J_star):
    """Efecto de que las mujeres se retiren a los 60 (vs 65), por pasos ACUMULADOS (el orden importa; es una descomposición, no una
    identidad): (a) ambos sexos a 65 [referencia = variante "retiro 65 para todos"]; (b) + S_m a los 60 (más sobrevivientes llegan);
    (c) + Ā_m hasta 59 (menos años de aporte); (d) + J* (más pasivas: mujeres 60-64) = modelo principal.  Devuelve TBP_C total en cada paso."""
    p = lambda S_m, A_m, J: tbp_c_sexos(N, sh_v, S_v, S_m, A_v, A_m, tau, rho, J)["TBP_C_total"]
    return pd.DataFrame({"a_ambos_65": p(S_m65, A_m65, J65), "b_mas_S_m60": p(S_m60, A_m65, J65),
                         "c_mas_Abar_m59": p(S_m60, A_m60, J65), "d_mas_J_star": p(S_m60, A_m60, J_star)})


# ================================================================== 6) MIGRACIÓN (corrección empírica implícita en la proyección ONU)
def factor_migracion(PM, PF, cohortes, N, sh_v, S_v, S_m, w_pob=None, edad_v=EDAD_V, edad_m=EDAD_M):
    """m_s = población ONU de edad r_s en el año t+r_s / (N_s·S_s)  (N_efectivo·S = población ONU).  Por sexo, con varones a edad_v y
    mujeres a edad_m.  `w_pob` (Serie por cohorte, opcional) repondera la población ONU con el escenario de natalidad (cohortes ≥2025),
    de modo que N_t (escenario) y la población usen la misma historia de nacimientos.  Mezcla migración neta, diferencias entre fuentes
    (DEIS vs ONU) y el desfase de nacimientos: NO es migración pura."""
    co = [int(c) for c in cohortes]
    pv = pd.Series([PM.loc[c + edad_v, edad_v] for c in co], index=co)
    pm = pd.Series([PF.loc[c + edad_m, edad_m] for c in co], index=co)
    if w_pob is not None:
        wv = w_pob.reindex(co).values
        pv, pm = pv * wv, pm * wv
    m_v = pv / (N * sh_v * S_v)
    m_m = pm / (N * (1 - sh_v) * S_m)
    return pd.DataFrame({"m_v": m_v, "m_m": m_m, "pob_v": pv, "pob_m": pm})


# ================================================================== 7) CONSTRUCCIÓN DEL MODELO COMPLETO (pasos acumulados 0-5)
def construir_modelo(d, s, tablas, PM, PF, srb, nat, abar_tot, tabla_mor, abar_sx=None, tau=TAU_DEF, rho_const=RHO_DEF,
                     A_obs=A_OBS_DEF, A_legal=A_LEGAL_DEF, escenario_nat="persistencia", escenario_mor="persiste",
                     col_abar_tot="A_bar_solo_asalariados", edad_retiro_mujeres=EDAD_M):
    """Pasos acumulados de TBP_C (cohortes de `d`):
      p0  base: Ā constante (14,2 y 30), ρ y τ constantes
      p1  + Ā(t) de la EPH (total, 18-64)
      p2  + ρ(t) ANSES (digitalizado; 'ultimo_valor' después de 2023; τ constante por supuesto)
      p3  + diferenciación por sexo y retiro (varones 65, mujeres 60): S_s(r_s), Ā_s hasta r_s−1, J* = varones 65+ y mujeres 60+
      (sensibilidad, aparte: *_m65 = "retiro a los 65 para todos" con J = J65, modelo base)
      p4  + escenario de natalidad (sección 9) y J* reponderado (sección 10) para cohortes ≥2025
      p5  + ajuste por migración (opcional, aparte)
    Si `abar_sx` es None (sin microdatos EPH por sexo) se calculan solo p0-p2 y nada se inventa para el resto.
    Devuelve dict con R (DataFrame por cohorte) y piezas auxiliares."""
    dd = d.set_index("cohorte")
    co = dd.index
    D = dependencias()
    R = pd.DataFrame(index=co)
    R["anio_65"] = co + 65
    N0 = dd["N_t"].astype(float)
    S65, J65 = dd["S65"], dd["J"]
    R["N_t"] = N0
    # ---- p0: base
    R["TBP_C_p0_A14_2"] = N0 * S65 * A_obs * tau / (J65 * rho_const)
    R["TBP_C_p0_A30"] = N0 * S65 * A_legal * tau / (J65 * rho_const)
    # ---- p1: Ā(t) EPH
    A_eph = abar_tot[col_abar_tot].reindex(co)
    assert A_eph.notna().all(), "Ā EPH no cubre todas las cohortes"
    R["A_eph"] = A_eph
    R["TBP_C_p1"] = N0 * S65 * A_eph * tau / (J65 * rho_const)
    # ---- p2: ρ(t) ANSES (+ τ constante), vía tbp_anses.asignar_rho_tau_por_cohorte (regla 'ultimo_valor')
    sr = tabla_mor.reset_index()[["anio", "rho", "rho_sin_moratoria"]]
    st = D.serie_tau(valor=tau, anios=sr["anio"].tolist())
    asig = D.asignar_rho_tau_por_cohorte(d, sr, st, edad_retiro=65, extrapolacion="ultimo_valor")
    R["rho_t"] = asig.set_index("cohorte")["rho_v"]
    R["tau_t"] = asig.set_index("cohorte")["tau_v"]
    R["TBP_C_p2"] = N0 * S65 * A_eph * R["tau_t"] / (J65 * R["rho_t"])
    # ρ por escenario de moratorias (persiste == p2, verificado)
    RHO_ESC = {}
    for esc in ESCENARIOS_MOR:
        rr, _ = rho_por_escenario(co + 65, tabla_mor, esc)
        RHO_ESC[esc] = pd.Series(rr.values, index=co)
    assert np.allclose(RHO_ESC["persiste"].values, R["rho_t"].values, atol=1e-9), "escenario 'persiste' debe coincidir con 'ultimo_valor'"
    rho_v = RHO_ESC[escenario_mor]
    R["rho_escenario"] = rho_v
    aux = dict(RHO_ESC=RHO_ESC, asig=asig, tabla_mor=tabla_mor)
    if abar_sx is None:
        return dict(R=R, **aux, con_sexo=False)

    # ---- sexo: share, N_s, S_s, E_s  (principal: varones 65, mujeres 60; sensibilidad: todos a 65 con J = J65)
    sh_v = share_varones(srb).reindex(co)
    assert sh_v.notna().all()
    R["share_varones"] = sh_v
    Sx = supervivencia_retiro(co, tablas, edad_v=EDAD_V, edad_m=edad_retiro_mujeres)
    Ex = er_retiro(co, tablas, edad_v=EDAD_V, edad_m=edad_retiro_mujeres)
    for c_ in Sx.columns:
        R[c_] = Sx[c_]
    for c_ in Ex.columns:
        R[c_] = Ex[c_]
    S_m65 = Sx["S_m65"]
    E_m65 = er_retiro(co, tablas, edad_v=EDAD_V, edad_m=EDAD_ALT)["E_m"]
    R["E_m65"] = E_m65
    for c_ in abar_sx.columns:
        R[c_] = abar_sx[c_].reindex(co)
    A_v = R["A_varones_65"]
    A_m = R[f"A_mujeres_{edad_retiro_mujeres}"]
    A_m65 = R["A_mujeres_65"]
    JJ = j_pasivos(PM, PF, co + 65, edad_m=edad_retiro_mujeres)
    JJ.index = co
    R["J65"] = JJ["J65"]; R["J_star"] = JJ["J_star"]
    assert np.allclose(R["J65"].values, J65.values, rtol=1e-9), "J65 (suma de PopMale+PopFemale 65+) debe reproducir la columna J de d"
    # ---- p3: sexo y retiro (varones 65, mujeres edad_m); J* = varones 65+ y mujeres edad_m+
    p3 = tbp_c_sexos(N0, sh_v, Sx["S_v"], Sx["S_m"], A_v, A_m, tau, rho_v, JJ["J_star"])
    R["TBP_C_p3"] = p3["TBP_C_total"]
    R["TBP_C_p3_J65"] = tbp_c_sexos(N0, sh_v, Sx["S_v"], Sx["S_m"], A_v, A_m, tau, rho_v, J65)["TBP_C_total"]       # misma numeración, J = 65+ total
    # ---- p4: escenario de natalidad y J* reponderado (solo cohortes ≥ 2025)
    w = pesos_cohorte_ma(s, nat, escenario_nat)
    nuevo = co >= 2025
    N4 = N0.copy()
    N4.loc[2025:2035] = nat.loc[2025:2035, escenario_nat].values

    def _j4(edad_m_, w_=w):
        j0 = j_pasivos(PM, PF, co + 65, edad_m=edad_m_); j0.index = co
        jr = j_pasivos(PM, PF, co + 65, edad_m=edad_m_, w=w_); jr.index = co
        j4 = j0["J_star"].copy(); j4[nuevo] = jr["J_star"][nuevo]
        return j4, jr
    J4, JJ_rw = _j4(edad_retiro_mujeres)
    J4_65, _ = _j4(EDAD_ALT)                                  # J = 65+ total reponderado (modelo base)
    R["N_4"] = N4; R["J_star_4"] = J4
    p4 = tbp_c_sexos(N4, sh_v, Sx["S_v"], Sx["S_m"], A_v, A_m, tau, rho_v, J4)
    R["TBP_C_p4"] = p4["TBP_C_total"]
    R["J_consistente"] = ~(co + EDAD_V - edad_retiro_mujeres > 2035) | (co < 2025)     # J* de mujeres en t+65 incluye cohortes hasta t+(65-edad_m)
    # ---- p5: migración (sobre p4)
    w_pob = pd.Series(1.0, index=w.index); w_pob.loc[2025:2035] = w.loc[2025:2035]       # solo cohortes ≥2025 (escenario)
    mig = factor_migracion(PM, PF, co, N4, sh_v, Sx["S_v"], Sx["S_m"], w_pob=w_pob, edad_m=edad_retiro_mujeres)
    R["m_v"], R["m_m"] = mig["m_v"], mig["m_m"]
    p5 = tbp_c_sexos(N4, sh_v, Sx["S_v"], Sx["S_m"], A_v, A_m, tau, rho_v, J4, mig["m_v"], mig["m_m"])
    R["TBP_C_p5"] = p5["TBP_C_total"]
    R["m_total"] = (mig["pob_v"] + mig["pob_m"]) / (p4["sob_v"] + p4["sob_m"])
    # ---- salidas por sexo (modelo p4) y TBP_A / TBP_B
    for c_ in ("TBP_C_varones", "TBP_C_mujeres", "TBP_C_total"):
        R[c_ + "_p4"] = p4[c_]
    ab = tbp_ab_sexos(p4, A_v, A_m, Ex["E_v"], Ex["E_m"], tau, rho_v, J4)
    for c_ in ab.columns:
        R[c_ + "_p4"] = ab[c_]
    # ---- ρ escenarios sobre p4
    for esc, rv in RHO_ESC.items():
        R[f"TBP_C_p4_rho_{esc}"] = tbp_c_sexos(N4, sh_v, Sx["S_v"], Sx["S_m"], A_v, A_m, tau, rv, J4)["TBP_C_total"]
    # ---- escenarios de natalidad (p4 con cada uno)
    for e_ in ESC_NAT:
        w_e = pesos_cohorte_ma(s, nat, e_)
        Ne = N0.copy(); Ne.loc[2025:2035] = nat.loc[2025:2035, e_].values
        Je, _ = _j4(edad_retiro_mujeres, w_e)
        R[f"TBP_C_p4_{e_}"] = tbp_c_sexos(Ne, sh_v, Sx["S_v"], Sx["S_m"], A_v, A_m, tau, rho_v, Je)["TBP_C_total"]
    # ---- SENSIBILIDAD (variante alternativa): "retiro a los 65 para todos" con J = población 65+ total (modelo base)
    s3 = tbp_c_sexos(N0, sh_v, Sx["S_v"], S_m65, A_v, A_m65, tau, rho_v, J65)
    s4 = tbp_c_sexos(N4, sh_v, Sx["S_v"], S_m65, A_v, A_m65, tau, rho_v, J4_65)
    R["TBP_C_p3_m65"] = s3["TBP_C_total"]; R["TBP_C_p4_m65"] = s4["TBP_C_total"]
    for c_ in ("TBP_C_varones", "TBP_C_mujeres"):
        R[c_ + "_p4_m65"] = s4[c_]
    ab65 = tbp_ab_sexos(s4, A_v, A_m65, Ex["E_v"], E_m65, tau, rho_v, J4_65)
    for c_ in ab65.columns:
        R[c_ + "_p4_m65"] = ab65[c_]
    desc = descomponer_retiro_mujeres(N0, sh_v, Sx["S_v"], S_m65, Sx["S_m"], A_v, A_m65, A_m, tau, rho_v, J65, JJ["J_star"])
    aux.update(p3=p3, p4=p4, p5=p5, s3=s3, s4=s4, ab=ab, ab65=ab65, mig=mig, desc=desc, w=w, JJ=JJ, JJ_rw=JJ_rw, Sx=Sx, Ex=Ex, N4=N4, J4=J4,
               sh_v=sh_v, rho_v=rho_v, tau=tau, A_v=A_v, A_m=A_m, A_m65=A_m65, edad_m=edad_retiro_mujeres)
    return dict(R=R, **aux, con_sexo=True)


# ================================================================== 8) CONTROLES (asserts) CON LOS DATOS REALES
def controles_modelo_avanzado(res, d, verbose=True):
    """Asserts del modelo ya construido (datos reales). Devuelve lista de mensajes OK."""
    R, msgs = res["R"], []
    dd = d.set_index("cohorte")
    tau, rho_c = res.get("tau", TAU_DEF), RHO_DEF
    # p0 reproduce las columnas de d
    assert np.allclose(R["TBP_C_p0_A14_2"], dd["TBP_C_obs"], rtol=1e-9) and np.allclose(R["TBP_C_p0_A30"], dd["TBP_C_legal"], rtol=1e-9)
    msgs.append("p0 reproduce TBP_C_obs (Ā=14,2) y TBP_C_legal (Ā=30) de `d`")
    tm = res["tabla_mor"]
    assert tm["rho_mor_implicito"].between(0, 1).all() and (tm["rho_mor_implicito"] < tm["rho_sin_moratoria"]).all()
    msgs.append(f"ρ_mor implícito ∈ (0,1) y < ρ sin moratoria en los {len(tm)} años (min {tm.rho_mor_implicito.min():.3f}, max {tm.rho_mor_implicito.max():.3f})")
    # ordenamiento por escenario: mayor ρ => menor TBP_C (solo cohortes con retiro > 2023, donde los escenarios difieren)
    if res["con_sexo"]:
        post = R.index + 65 > 2023
        a, b_, c_ = (R.loc[post, f"TBP_C_p4_rho_{k}"] for k in ("persiste", "mitad", "sin_moratorias"))
        assert (a > b_).all() and (b_ > c_).all(), "mayor ρ debe dar menor TBP_C"
        pre = ~post
        assert np.allclose(R.loc[pre, "TBP_C_p4_rho_persiste"], R.loc[pre, "TBP_C_p4_rho_sin_moratorias"]), "antes de 2024 los escenarios coinciden"
        msgs.append("escenarios de ρ: TBP_C(persiste) > TBP_C(mitad) > TBP_C(sin moratorias) en todas las cohortes con retiro > 2023; idénticos antes")
        Sx = res["Sx"]
        assert np.allclose(R["N_t"] * R["share_varones"] + R["N_t"] * (1 - R["share_varones"]), R["N_t"]) and R["share_varones"].between(0.45, 0.55).all()
        assert np.allclose(res["p3"]["N_v"] + res["p3"]["N_m"], R["N_t"])
        msgs.append("N_varones + N_mujeres = N_t; share_varones ∈ (0,45; 0,55)")
        assert Sx[["S_v", "S_m", "S_m65", "S_tot65"]].apply(lambda c: c.between(0, 1, inclusive="neither").all()).all()
        assert np.allclose(Sx["S_tot65"], dd["S65"], atol=1e-12)
        msgs.append("S_s(r_s) ∈ (0,1); S_Total(65) cohortal reproduce `S65` de `d`")
        p3 = res["p3"]
        indep = (p3["N_v"] * Sx["S_v"] * R["A_varones_65"] + p3["N_m"] * Sx["S_m"] * res["A_m"]) * tau / (R["J_star"] * res["rho_v"])
        assert np.allclose(p3["TBP_C_varones"] + p3["TBP_C_mujeres"], p3["TBP_C_total"]) and np.allclose(indep, R["TBP_C_p3"])
        msgs.append("TBP_C_varones + TBP_C_mujeres = TBP_C_total (calculado también por fuera como Σ_s N_s S_s Ā_s τ /(J*ρ))")
        # control de reducción al modelo base: share=1 varón, r=65, Ā constante, S = S65 de d, J = J de d
        base = tbp_c_sexos(R["N_t"], 1.0, dd["S65"], dd["S65"], 30.0, 30.0, TAU_DEF, RHO_DEF, dd["J"])
        assert np.allclose(base["TBP_C_total"], dd["TBP_C_legal"], rtol=1e-9) and np.allclose(base["TBP_C_mujeres"], 0)
        msgs.append("con share=1 varón, r=65, Ā=30 constante, ρ y τ constantes el modelo reproduce TBP_C_legal de `d`")
        # linealidad
        k = 1.7
        x1 = tbp_c_sexos(R["N_t"], res["sh_v"], Sx["S_v"], Sx["S_m"], R["A_varones_65"], res["A_m"], tau, res["rho_v"], R["J_star"])["TBP_C_total"]
        xa = tbp_c_sexos(R["N_t"], res["sh_v"], Sx["S_v"], Sx["S_m"], k * R["A_varones_65"], k * res["A_m"], tau, res["rho_v"], R["J_star"])["TBP_C_total"]
        xt = tbp_c_sexos(R["N_t"], res["sh_v"], Sx["S_v"], Sx["S_m"], R["A_varones_65"], res["A_m"], k * tau, res["rho_v"], R["J_star"])["TBP_C_total"]
        assert np.allclose(xa, k * x1) and np.allclose(xt, k * x1)
        msgs.append("linealidad: TBP_C escala exactamente con Ā y con τ (factor 1,7)")
        # p4 = p3 para cohortes ≤ 2024 ; migración sin ajuste = p4
        assert np.allclose(R.loc[R.index <= 2024, "TBP_C_p4"], R.loc[R.index <= 2024, "TBP_C_p3"])
        msgs.append("p4 = p3 para cohortes ≤ 2024 (solo cambian las ≥ 2025)")
        pb = res["p5"]
        assert np.allclose(pb["sob_v"] + pb["sob_m"], res["mig"]["pob_v"] + res["mig"]["pob_m"]), "N_efectivo·S debe igualar la población ONU"
        sin = tbp_c_sexos(res["N4"], res["sh_v"], Sx["S_v"], Sx["S_m"], R["A_varones_65"], res["A_m"], tau, res["rho_v"], res["J4"], 1.0, 1.0)
        assert np.allclose(sin["TBP_C_total"], R["TBP_C_p4"])
        msgs.append("migración: N_efectivo·S = población ONU a la edad de retiro; con m=1 vuelve exactamente a p4")
        ab = res["ab"]
        assert np.allclose(ab["TBP_A_total"], ab["D_star"] * ab["TBP_B_total"])
        msgs.append("identidad TBP_A_total = D*·TBP_B_total (agregación: TBP_A aditivo, TBP_B ponderado por sobrevivientes)")
    if verbose:
        for m_ in msgs:
            print("OK:", m_)
    return msgs


# ================================================================== 9) AUTOTEST CON DATOS SINTÉTICOS
def _banner_ma(txt="DATOS SINTÉTICOS — no son Argentina"):
    line = "#" * 78
    print("\n".join([line, "##" + txt.center(74) + "##", line]))


def autotest_modelo_avanzado(verbose=True):
    """Pruebas de las funciones puras con datos SINTÉTICOS (inventados, solo para verificar código; nunca se usan en salidas reales)."""
    if verbose:
        _banner_ma()
    co = pd.Index(np.arange(2000, 2006), name="cohorte")
    # población sintética: 120 mil personas por edad y sexo en todos los años, varones algo menos
    yrs = np.arange(2060, 2075)
    PM = pd.DataFrame(100000.0, index=yrs, columns=np.arange(0, 101))
    PF = pd.DataFrame(120000.0, index=yrs, columns=np.arange(0, 101))
    # J*: varones 65+ (36 edades, incl. 100) + mujeres 60+ (41 edades)
    J = j_pasivos(PM, PF, [2065], edad_m=60)
    assert abs(J.loc[2065, "J_v"] - 36 * 1e5) < 1e-6 and abs(J.loc[2065, "J_m"] - 41 * 1.2e5) < 1e-6
    assert abs(J.loc[2065, "J65"] - 36 * 2.2e5) < 1e-6
    assert abs((J.loc[2065, "J_star"] - J.loc[2065, "J65"]) - 5 * 1.2e5) < 1e-6          # difieren en las 5 edades 60-64 de mujeres
    # reponderación: w=0,5 para todas las cohortes -> J a la mitad
    w = pd.Series(0.5, index=np.arange(1900, 2101))
    assert abs(j_pasivos(PM, PF, [2065], edad_m=60, w=w).loc[2065, "J_star"] - 0.5 * J.loc[2065, "J_star"]) < 1e-6
    # share por SRB
    assert abs(share_varones(105.0) - 105.0 / 205.0) < 1e-15 and abs(share_varones(100.0) - 0.5) < 1e-15
    # ρ implícito: ida y vuelta
    s_ = np.array([0.3, 0.5, 0.7]); rm = np.array([0.25, 0.30, 0.35]); rs = np.array([0.5, 0.55, 0.6])
    rt = s_ * rm + (1 - s_) * rs
    assert np.allclose(rho_implicito(rt, rs, s_), rm)
    # tabla y escenarios
    sr = pd.DataFrame({"anio": [2021, 2022, 2023], "rho": rt, "rho_sin_moratoria": rs})
    be = pd.DataFrame({"anio": [2021, 2022, 2023], "share_moratoria": s_})
    tm = tabla_rho_moratoria(sr, be)
    r_p, _ = rho_por_escenario([2022, 2030], tm, "persiste"); r_h, _ = rho_por_escenario([2030], tm, "mitad"); r_0, _ = rho_por_escenario([2030], tm, "sin_moratorias")
    assert abs(r_p[2022] - rt[1]) < 1e-12 and abs(r_p[2030] - rt[2]) < 1e-12 and abs(r_0[2030] - rs[2]) < 1e-12
    assert r_p[2030] < r_h[2030] < r_0[2030]                                    # sin moratorias => ρ más alto
    try:
        rho_por_escenario([2030], tm, "otro"); raise AssertionError("debía fallar")
    except ValueError:
        pass
    # TBP_C por sexo
    N = pd.Series(1e6, index=co); Sv = pd.Series(0.85, index=co); Sm = pd.Series(0.93, index=co)
    Jc = pd.Series(1e7, index=co)
    a = tbp_c_sexos(N, 0.5, Sv, Sm, 10.0, 8.0, 0.2, 0.4, Jc)
    assert np.allclose(a["N_v"] + a["N_m"], N) and np.allclose(a["TBP_C_varones"] + a["TBP_C_mujeres"], a["TBP_C_total"])
    assert np.allclose(a["TBP_C_total"], (5e5 * 0.85 * 10 + 5e5 * 0.93 * 8) * 0.2 / (1e7 * 0.4))
    b = tbp_c_sexos(N, 1.0, Sv, Sv, 30.0, 30.0, 0.2177, 0.403, Jc)              # control: share=1 varón == fórmula base
    assert np.allclose(b["TBP_C_total"], N * Sv * 30 * 0.2177 / (Jc * 0.403)) and np.allclose(b["TBP_C_mujeres"], 0)
    assert np.allclose(tbp_c_sexos(N, 0.5, Sv, Sm, 20.0, 16.0, 0.2, 0.4, Jc)["TBP_C_total"], 2 * a["TBP_C_total"])      # linealidad en Ā
    assert np.allclose(tbp_c_sexos(N, 0.5, Sv, Sm, 10.0, 8.0, 0.4, 0.4, Jc)["TBP_C_total"], 2 * a["TBP_C_total"])       # linealidad en τ
    assert (tbp_c_sexos(N, 0.5, Sv, Sm, 10.0, 8.0, 0.2, 0.8, Jc)["TBP_C_total"] < a["TBP_C_total"]).all()               # ρ mayor => menor
    # TBP_A y TBP_B por sexo: identidad TBP_A = D*·TBP_B
    ab = tbp_ab_sexos(a, 10.0, 8.0, pd.Series(17.0, index=co), pd.Series(26.0, index=co), 0.2, 0.4, Jc)
    assert np.allclose(ab["TBP_A_total"], ab["D_star"] * ab["TBP_B_total"])
    assert np.allclose(ab["TBP_A_total"], a["TBP_C_varones"] / 17.0 + a["TBP_C_mujeres"] / 26.0)
    # migración: N_efectivo·S = población
    mg = factor_migracion(pd.DataFrame({65: [3e5, 3e5, 3e5, 3e5, 3e5, 3e5]}, index=co + 65), pd.DataFrame({60: [3.3e5] * 6}, index=co + 60),
                          co, N, 0.5, Sv, Sm, edad_m=60)
    assert np.allclose(mg["m_v"] * N * 0.5 * Sv, 3e5) and np.allclose(mg["m_m"] * N * 0.5 * Sm, 3.3e5)
    c = tbp_c_sexos(N, 0.5, Sv, Sm, 10.0, 8.0, 0.2, 0.4, Jc, mg["m_v"], mg["m_m"])
    assert np.allclose(c["sob_v"] + c["sob_m"], 3e5 + 3.3e5)
    # descomposición: pasos coherentes (a: 65 ambos; d = p3)
    de = descomponer_retiro_mujeres(N, 0.5, Sv, pd.Series(0.9, index=co), Sm, 10.0, 8.0, 7.0, 0.2, 0.4, Jc, Jc * 1.1)
    assert np.allclose(de["d_mas_J_star"], tbp_c_sexos(N, 0.5, Sv, Sm, 10.0, 7.0, 0.2, 0.4, Jc * 1.1)["TBP_C_total"])
    if verbose:
        print("autotest OK (J* y reponderación, share SRB, ρ_mor implícito y escenarios, TBP_C por sexo, linealidad, control share=1, "
              "TBP_A/TBP_B y D*, migración, descomposición). Datos SINTÉTICOS: no son Argentina.")
        _banner_ma("FIN AUTOTEST — estos números NO son Argentina")
    return True

### 15.2 Carga de datos y Ā por sexo

Se cargan: población ONU por edad simple y **sexo** (`PopMale`, `PopFemale`), la razón de masculinidad al nacer `SRB` (ONU), los escenarios de natalidad de la sección 9, el $\bar A$ total de la EPH (sección 12) y las tablas de mortalidad por sexo. Para $\bar A$ **por sexo** se corre `pipeline(..., sexo=1|2)` sobre la EPH **real** de `datos/eph/` (o la que ya dejó la sección 12 en memoria) y se **trunca la suma de edades** a $r_s-1$. Si no hay microdatos y tampoco el CSV derivado `abar_eph_por_sexo_cohorte.csv`, **se omite la parte por sexo** (no se inventa nada): quedan los pasos 0–2 y la sección de $\rho$.

La mortalidad de Gompertz que usa la EPH para $S(a)$, $a<r_s$, se calibra con la $S(65)$ **del sexo**. $\bar A$ cubre sólo a asalariados con descuento jubilatorio (cota inferior).

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

CANDS_15 = ["../datos/procesados", "datos/procesados", DATA]
D15 = dependencias()
TAB15 = D15.cargar_tablas(rutas=CANDS_15)                       # tablas ONU abreviadas Total / Male / Female
CO15 = [int(c) for c in d.cohorte]
PM15, PF15 = cargar_poblacion_sexo(rutas=CANDS_15)              # personas por edad simple y año
SRB15 = cargar_srb(rutas=CANDS_15)                              # varones por 100 mujeres al nacer (ONU)
NAT15 = cargar_natalidad_escenarios(rutas=CANDS_15)             # escenarios sección 9
ABAR_TOT15 = leer_csv_ma("abar_eph_por_cohorte.csv", CANDS_15, destino=OUT).set_index("cohorte")   # Ā total EPH (sección 12)
SERIE_RHO15 = D15.serie_rho_digitalizada(rutas=CANDS_15)
BEN15 = D15.serie_beneficiarios_digitalizada()
TAB_MOR15 = tabla_rho_moratoria(SERIE_RHO15, BEN15)
print("SRB ONU (varones por 100 mujeres): min %.1f | max %.1f | share varones 2000 = %.4f" % (SRB15.min(), SRB15.max(), share_varones(SRB15[2000])))

# --- Ā por sexo: EPH real si está; si no, CSV derivado (si existe); si no, se omite
DEST15 = next((p for p in ["../datos/procesados", "datos/procesados"] if os.path.isdir(p)), OUT)
F_SEXO15 = "abar_eph_por_sexo_cohorte.csv"
df_eph15 = globals().get("df_eph")
if df_eph15 is None:
    srcs15 = D15.find_eph_files(["datos/eph", "../datos/eph", "../../datos/eph"])
    df_eph15 = D15.read_all(srcs15) if srcs15 else None
ABAR_SX15, ORIGEN_SEXO15 = None, None
if df_eph15 is not None:
    ABAR_SX15 = abar_por_sexo(df_eph15, TAB15, CO15)
    ORIGEN_SEXO15 = "EPH real (pipeline por sexo sobre datos/eph)"
    os.makedirs(DEST15, exist_ok=True)
    ABAR_SX15.round(6).reset_index().to_csv(os.path.join(DEST15, F_SEXO15), index=False)
else:
    pc = next((os.path.join(c, F_SEXO15) for c in CANDS_15 if os.path.exists(os.path.join(c, F_SEXO15))), None)
    if pc is not None:
        ABAR_SX15 = pd.read_csv(pc).set_index("cohorte"); ORIGEN_SEXO15 = f"CSV derivado de la EPH real ({pc})"
HAY_SEXO15 = ABAR_SX15 is not None
if HAY_SEXO15:
    print("Ā por sexo:", ORIGEN_SEXO15)
    display(ABAR_SX15.loc[[1950, 1960, 1980, 2000, 2024, 2035], [c for c in ABAR_SX15.columns if c.startswith("A_")]].round(2))
else:
    print("⚠️ NO hay microdatos EPH ni CSV por sexo: se omite la parte por sexo (pasos 3-5). El CSV abar_eph_por_cohorte.csv NO tiene sexo y NO se usa para inventarlo.")

### 15.3 Sexo y edad de retiro: el modelo y la agregación

**Paso a paso (cada uno con su porqué):**

1. **Reparto por sexo.** $share_{varones}(t)=SRB_t/(100+SRB_t)$ con el $SRB$ de la ONU del año de nacimiento (≈105 varones por 100 mujeres ⇒ 51,2 %). No se usa 51/49 fijo. $N_{v}=N_t\,share$, $N_m=N_t-N_v$ (suman $N_t$ exacto).
2. **Supervivencia al retiro** $S_s(r_s)$: tabla ONU **del sexo** siguiendo la trayectoria de la cohorte (misma regla que $1-\mu$ de la sección 1; se reusa `supervivencia_cohortal`). Las mujeres llegan a los 60 con mucha más probabilidad que a los 65, y que los varones a los 65.
3. **$\bar A_s$**: densidad de aportes de la EPH **del sexo** sumada hasta $r_s-1$ (varones 18–64, mujeres 18–59). Retirarse antes recorta años de aporte.
4. **Denominador $J^*\rho$:** $J^*$ = varones 65+ **más** mujeres 60+ en el año $t+65$ (población ONU por edad simple). Si las mujeres cobran desde los 60, sus 60–64 años son **pasivas**; contarlas como aportantes (modelo base: todos a 65) **sobrecuenta aportes y omite pasivas**.
5. **Agregación (una sola, definida).** *Se SUMAN* las contribuciones al numerador: $TBP_{C}^{total}=TBP_C^{varones}+TBP_C^{mujeres}$, ambos sobre el **mismo** $J^*\rho$ (por eso la suma es el total). *Se PROMEDIAN, ponderando por los sobrevivientes al retiro* $N_sS_s$, las intensidades por persona ($\bar A$, $E_r$, $TBP_B$). $TBP_A^{total}=\sum_s TBP_C^s/E_{r,s}$ y vale $TBP_A=D^*\cdot TBP_B$ con $D^*=\sum_s N_sS_s/J^*$.
6. **$E_r$ por sexo y edad de retiro:** `er_cohortal(edad=65, 'Male')` y `er_cohortal(edad=60, 'Female')` con las tablas ONU por sexo (nada de factores 0,92/1,08).

$\rho$ y $\tau$ son **los mismos para ambos sexos** y del año $t+65$ (no hay series por sexo; las mujeres se retiran 5 años antes, así que es una simplificación).

In [ ]:
RES15 = construir_modelo(d, s, TAB15, PM15, PF15, SRB15, NAT15, ABAR_TOT15, TAB_MOR15, abar_sx=ABAR_SX15, tau=TAU, rho_const=RHO,
                         A_obs=A_OBS, A_legal=A_LEGAL, escenario_nat="persistencia", escenario_mor="persiste")
R15 = RES15["R"]
SEL15 = [1955, 1960, 1980, 2000, 2024, 2035]
if HAY_SEXO15:
    cols = ["share_varones", "S_v", "S_m", "S_m65", "E_v", "E_m", "E_m65", "A_varones_65", "A_mujeres_60", "A_mujeres_65", "J65", "J_star"]
    display(R15.loc[SEL15, cols].round(3))
    print("Sobrevivientes al retiro y TBP_C por sexo (modelo completo, paso 4; ambos sobre el mismo J*·ρ):")
    display(R15.loc[SEL15, ["TBP_C_varones_p4", "TBP_C_mujeres_p4", "TBP_C_total_p4", "TBP_A_varones_p4", "TBP_A_mujeres_p4", "TBP_A_total_p4",
                            "TBP_B_varones_p4", "TBP_B_mujeres_p4", "TBP_B_total_p4", "D_star_p4"]].round(4))
    print("Efecto de que las mujeres se retiren a los 60 en vez de 65 (TBP_C, sin natalidad ni migración; pasos ACUMULADOS, el orden importa):")
    display(RES15["desc"].loc[SEL15].round(4))
    print("  a: todos a 65 (J = 65+ total, modelo base)  ->  b: + más mujeres llegan a los 60 (S_m(60) > S_m(65))  ->  c: + Ā hasta 59 (menos años de aporte)  ->  d: + J* (60-64 pasivas)")
    print("TBP_C: variante J* vs variante J = 65+ total (mismo numerador), cohortes", SEL15)
    display(R15.loc[SEL15, ["TBP_C_p3", "TBP_C_p3_J65"]].assign(**{"Δ% J* vs J65": lambda x: 100 * (x.TBP_C_p3 / x.TBP_C_p3_J65 - 1)}).round(4))

In [ ]:
if HAY_SEXO15:
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
    co_ = R15.index
    ax[0].stackplot(co_, R15.TBP_C_varones_p4, R15.TBP_C_mujeres_p4, colors=["#1f3a5f", "#d55e00"], labels=["varones (retiro 65)", "mujeres (retiro 60)"], alpha=.9)
    ax[0].set_title("TBP_C por sexo (modelo completo, paso 4)\nla suma de ambos es el total (mismo J*·ρ)"); ax[0].legend(fontsize=8, loc="upper right")
    ax[1].plot(co_ + 65, R15.J65 / 1e6, color="#444444", ls="--", label="J = 65+ total (modelo base)")
    ax[1].plot(co_ + 65, R15.J_star / 1e6, color="#0072b2", lw=2, label="J* = varones 65+ y mujeres 60+")
    ax[1].set_title("Pasivos potenciales (millones), año calendario t+65"); ax[1].set_xlabel("año calendario"); ax[1].legend(fontsize=8)
    ax[2].plot(co_, R15.TBP_C_p4_m65, color="#444444", ls="--", label="SENSIBILIDAD: todos a 65, J = 65+ total")
    ax[2].plot(co_, R15.TBP_C_p4, color="#0072b2", lw=2, label="PRINCIPAL: varones 65 / mujeres 60, J*")
    ax[2].set_title("TBP_C: retiro diferenciado vs 'todos a 65'"); ax[2].legend(fontsize=8)
    for a_ in ax:
        a_.grid(alpha=.3); a_.axvline(2024.5, color="gray", ls=":", lw=1)
    ax[0].set_xlabel("cohorte"); ax[2].set_xlabel("cohorte"); ax[0].set_ylabel("años de aporte por año de jubilado")
    fig.suptitle("Sección 15.3: sexo y edad de retiro (cohortes ≥2025: escenario 'persistencia' de natalidad; niveles condicionados a supuestos)", fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.savefig(f"{OUT}/seccion_15_sexo_retiro.png", dpi=130); plt.show()

### 15.4 $\rho$ con moratorias (escenario, por cohorte)

**Idea.** $\rho$ = haber medio / salario imponible medio. El haber medio de **todos** los beneficiarios es el promedio ponderado por beneficiarios del haber con moratoria y sin moratoria; como el denominador (salario imponible) es el mismo,
$$\rho_{total}=s\,\rho_{mor}+(1-s)\,\rho_{sin\,mor}\ \Rightarrow\ \rho_{mor}=\frac{\rho_{total}-(1-s)\,\rho_{sin\,mor}}{s}.$$
Con $\rho_{total}$, $\rho_{sin\,mor}$ y $s$ de ANSES (Gráficos 4.3 y 3.1, **digitalizados, precisión aprox.**) se despeja $\rho_{mor}$ implícito por año y se verifica que esté en $(0,1)$ y por debajo del $\rho$ sin moratoria. Es una aproximación: depende de que el "sin moratoria" del Anuario comparta el denominador y de la digitalización.

**Escenarios para cohortes que se retiran después de 2023** (**supuestos a validar por el grupo; no predicciones ni hechos**): $\rho=s_{esc}\,\rho_{mor}(2023)+(1-s_{esc})\,\rho_{sin\,mor}(2023)$ con
* **persiste:** $s$ = 62,9 % (la participación observada en 2023; coincide con la regla `ultimo_valor` de la sección 13, $\rho=0{,}403$);
* **mitad:** $s\approx 31\,\%$;
* **sin moratorias:** $s=0$ ⇒ $\rho=\rho_{sin\,mor}(2023)=56{,}3\,\%$.

Hasta el retiro de 2023 todos los escenarios usan el $\rho$ observado. $s$ es un stock de beneficiarios, no la participación en las altas nuevas; **no se inventa cobertura real**. Un $\rho$ más alto **reduce** $TBP_C$ (más pasivo por aportante).

In [ ]:
display(TAB_MOR15.round(4).rename(columns={"rho": "ρ_total", "rho_sin_moratoria": "ρ_sin_mor", "share_moratoria": "s (benef. moratoria)", "rho_mor_implicito": "ρ_mor implícito"}))
print("ρ_mor implícito: min %.3f | max %.3f | siempre en (0,1) y por debajo de ρ_sin_mor: %s" % (TAB_MOR15.rho_mor_implicito.min(), TAB_MOR15.rho_mor_implicito.max(),
      bool((TAB_MOR15.rho_mor_implicito < TAB_MOR15.rho_sin_moratoria).all())))
RESUM_RHO15 = pd.DataFrame({e: RES15["RHO_ESC"][e] for e in ESCENARIOS_MOR}).loc[[1950, 1958, 1959, 1975, 2000, 2035]]
print("ρ por escenario según la cohorte (retiro en cohorte+65):"); display(RESUM_RHO15.round(4))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
a_ = TAB_MOR15
ax[0].plot(a_.index, 100 * a_.rho, color="#0072b2", lw=2, label="ρ total (SIPA)")
ax[0].plot(a_.index, 100 * a_.rho_sin_moratoria, color="#222222", lw=2, label="ρ sin moratoria")
ax[0].plot(a_.index, 100 * a_.rho_mor_implicito, color="#d55e00", lw=2, label="ρ con moratoria (implícito)")
for e, col in zip(ESCENARIOS_MOR, ("#0072b2", "#009e73", "#cc79a7")):
    rho_e = RES15["RHO_ESC"][e].loc[1959:]; anios_ = rho_e.index + 65
    ax[0].plot([2023, 2035], [100 * rho_e.iloc[0]] * 2, ls="--", color=col, lw=1.5, label=f"supuesto post-2023: {ETQ_MOR[e]}")
ax[0].axvline(2023.5, color="gray", ls=":"); ax[0].set_ylim(25, 70); ax[0].grid(alpha=.3); ax[0].legend(fontsize=7.5, loc="lower right")
ax[0].set_title("ρ por año (2009-2023 digitalizado, aprox.) y supuestos desde 2024"); ax[0].set_ylabel("%"); ax[0].set_xlabel("año de retiro")
if HAY_SEXO15:
    for e, col in zip(ESCENARIOS_MOR, ("#0072b2", "#009e73", "#cc79a7")):
        ax[1].plot(R15.index, R15[f"TBP_C_p4_rho_{e}"], color=col, lw=2, label=ETQ_MOR[e])
    ax[1].axvspan(1958.5, 2035.5, color="gray", alpha=.08); ax[1].text(0.97, 0.97, "sombreado: retiro ≥2024,\nρ = SUPUESTO", transform=ax[1].transAxes, ha="right", va="top", fontsize=8, color="dimgray")
    ax[1].set_title("TBP_C (modelo completo, paso 4) según escenario de moratorias"); ax[1].set_xlabel("cohorte"); ax[1].set_ylabel("años de aporte por año de jubilado")
    ax[1].legend(fontsize=8, loc="lower left"); ax[1].grid(alpha=.3)
fig.suptitle("Sección 15.4: ρ con moratorias como ESCENARIO (no predicción)", fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.savefig(f"{OUT}/seccion_15_rho_moratorias.png", dpi=130); plt.show()
if HAY_SEXO15:
    t_ = R15.loc[[1960, 1980, 2000, 2024, 2035], [f"TBP_C_p4_rho_{e}" for e in ESCENARIOS_MOR]]
    t_["Δ% mitad vs persiste"] = 100 * (t_.TBP_C_p4_rho_mitad / t_.TBP_C_p4_rho_persiste - 1)
    t_["Δ% sin_moratorias vs persiste"] = 100 * (t_.TBP_C_p4_rho_sin_moratorias / t_.TBP_C_p4_rho_persiste - 1)
    display(t_.round(4))

### 15.5 Ajuste por migración (opcional, aparte)

**Idea.** Se mide, para cada cohorte y sexo, el cociente entre la población ONU de la edad de retiro **en el año del retiro** y los sobrevivientes que implican los nacimientos propios:
$$m_s(t)=\frac{\text{Pob}_{ONU}(r_s,\ t+r_s)}{N_s\cdot S_s(r_s)},\qquad N_{efectivo}\cdot S=\text{Pob}_{ONU}$$
Si $m>1$ hay más gente de esa edad de la que implican los nacimientos y la mortalidad (inmigración neta, pero también diferencias entre fuentes). Con `ajuste_migracion=True` el numerador pasa a ser $\text{Pob}_{ONU}\cdot\bar A_s\cdot\tau$.

> ⚠️ **No es migración pura.** Mezcla migración neta, errores o diferencias entre fuentes (DEIS vs ONU: la cohorte 2024 tiene $m\approx1{,}22$ porque DEIS registra ~413 mil nacimientos y la ONU supone ~507 mil) y el desfase de nacimientos (la población "de edad 65" a mitad de año mezcla dos cohortes). Además supone que los migrantes aportaron como los nativos (Ā igual), lo cual no se sabe.
>
> **Cohortes ≥2025:** $N_t$ es el escenario de la sección 9 y la población ONU está construida con *otros* nacimientos. Se repondera la población con $w=N_{escenario}/N_{ONU}$ (como la sección 10); entonces $m=\text{Pob}_{ONU}/(N_{ONU}\,S)$ es el cociente interno de la ONU (≈1,0), **sin** el desfase DEIS/ONU. Es lo coherente, pero deja de "corregir" N.

In [ ]:
if HAY_SEXO15:
    display(R15.loc[SEL15, ["m_v", "m_m", "m_total", "TBP_C_p4", "TBP_C_p5"]].assign(**{"Δ% migración": lambda x: 100 * (x.TBP_C_p5 / x.TBP_C_p4 - 1)}).round(4))
    pre, post = R15.index <= 2024, R15.index >= 2025
    print(f"m total: media cohortes ≤2024 = {R15.m_total[pre].mean():.3f} (máx {R15.m_total[pre].max():.3f} en cohorte {R15.m_total[pre].idxmax()}) | media cohortes ≥2025 = {R15.m_total[post].mean():.3f}")
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
    ax[0].plot(R15.index, R15.m_v, color="#1f3a5f", label="varones (edad 65)"); ax[0].plot(R15.index, R15.m_m, color="#d55e00", label="mujeres (edad 60)")
    ax[0].plot(R15.index, R15.m_total, color="k", lw=2, label="total"); ax[0].axhline(1, color="gray", lw=.8)
    ax[0].axvline(2024.5, color="gray", ls=":"); ax[0].set_title("Factor m = población ONU / (N·S)"); ax[0].set_xlabel("cohorte"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    ax[1].plot(R15.index, R15.TBP_C_p4, color="#0072b2", lw=2, label="sin ajuste (paso 4)"); ax[1].plot(R15.index, R15.TBP_C_p5, color="#cc3311", lw=2, ls="--", label="con ajuste de migración (paso 5)")
    ax[1].axvline(2024.5, color="gray", ls=":"); ax[1].set_title("TBP_C con y sin ajuste de migración"); ax[1].set_xlabel("cohorte"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    fig.suptitle("Sección 15.5: corrección implícita en la proyección ONU (mezcla migración, fuentes y desfase de nacimientos)", fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.savefig(f"{OUT}/seccion_15_migracion.png", dpi=130); plt.show()

### 15.6 Comparación final: TBP_C por cohorte, paso a paso

Pasos **acumulados** (cada uno agrega un supuesto al anterior):

| paso | qué agrega |
|---|---|
| **0** | base: $\bar A$ constante (14,2 y 30), $\rho=0{,}403$, $\tau=0{,}2177$ constantes, $J$ = 65+ total |
| **1** | + $\bar A(t)$ de la EPH (total, asalariados, cota inferior) |
| **2** | + $\rho(t)$ ANSES (digitalizado 2009–2023; `ultimo_valor` después; $\tau$ constante) |
| **3** | + diferenciación por sexo y retiro (varones 65, mujeres 60) con $J^*$ |
| **4** | + escenario de natalidad "persistencia" (sección 9) y $J^*$ reponderado (sección 10) para cohortes ≥2025 |
| **5** | + ajuste de migración (opcional, aparte) |

Los pasos 1–5 usan $\bar A$ de la EPH, así que **no son comparables en nivel con la línea $\bar A=30$**: ésta es una referencia (requisito legal, no un máximo). Como $\rho_{ANSES}$ coincide con 0,403 desde la cohorte 1959, el paso 2 sólo se distingue en 1950–1958. **Condicional a supuestos; no es una predicción.** Para cohortes ≥2025 el paso 4 es consistente salvo que el $J^*$ de mujeres 60+ en $t+65$ incluye cohortes posteriores a 2035 (sin escenario) a partir de $t>2030$: ver `J_consistente` en el CSV.

In [ ]:
PASOS15 = [("TBP_C_p0_A30", "0  base, Ā=30 constante (referencia)", "#999999", ":", 1.5), ("TBP_C_p0_A14_2", "0  base, Ā=14,2 constante", "#444444", "--", 1.5),
           ("TBP_C_p1", "1  + Ā(t) EPH", "#0072b2", "-", 2), ("TBP_C_p2", "2  + ρ(t) ANSES", "#56b4e9", "-", 2)]
if HAY_SEXO15:
    PASOS15 += [("TBP_C_p3", "3  + sexo y retiro 65/60 (J*)", "#009e73", "-", 2), ("TBP_C_p4", "4  + natalidad y J* reponderado", "#e69f00", "-", 2),
                ("TBP_C_p5", "5  + migración (opcional)", "#cc3311", "--", 1.8)]
fig, ax = plt.subplots(1, 2, figsize=(16, 5.4))
for col, lab, c_, ls_, lw in PASOS15:
    ax[0].plot(R15.index, R15[col], color=c_, ls=ls_, lw=lw, label=lab)
ax[0].axvspan(1958.5, 2035.5, color="gray", alpha=.06); ax[0].axvline(2024.5, color="gray", ls=":", lw=1)
ax[0].set_title("TBP_C por cohorte 1950-2035, pasos acumulados"); ax[0].set_xlabel("año de nacimiento de la cohorte"); ax[0].set_ylabel("años de aporte por año de jubilado")
ax[0].legend(fontsize=8, loc="upper right"); ax[0].grid(alpha=.3)
ant = {"TBP_C_p1": "TBP_C_p0_A14_2", "TBP_C_p2": "TBP_C_p1", "TBP_C_p3": "TBP_C_p2", "TBP_C_p4": "TBP_C_p3", "TBP_C_p5": "TBP_C_p4"}
for col, lab, c_, ls_, lw in PASOS15:
    if col in ant:
        ax[1].plot(R15.index, 100 * (R15[col] / R15[ant[col]] - 1), color=c_, ls=ls_, lw=lw, label=lab.split("  ")[1] + (" (vs Ā=14,2)" if col == "TBP_C_p1" else ""))
ax[1].axhline(0, color="gray", lw=.8); ax[1].axvline(2024.5, color="gray", ls=":", lw=1)
ax[1].set_title("Efecto marginal de cada paso (% sobre el paso anterior)"); ax[1].set_xlabel("año de nacimiento de la cohorte"); ax[1].set_ylabel("%")
ax[1].legend(fontsize=8, loc="lower left"); ax[1].grid(alpha=.3)
fig.suptitle("Sección 15.6: TBP_C base vs modelo avanzado (niveles condicionados a supuestos; no es una predicción)", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.savefig(f"{OUT}/seccion_15_pasos_TBP_C.png", dpi=130); plt.show()

COLS_T15 = [c for c, *_ in PASOS15]
T15 = R15.loc[SEL15, COLS_T15].rename(columns={c: l.split("  ")[0] + " " + l.split("  ")[1] for c, l, *_ in PASOS15})
print("TBP_C por cohorte y paso (años):"); display(T15.round(4))
print("Δ% de cada paso respecto del anterior (paso 1 vs base Ā=14,2):")
display(pd.DataFrame({col: 100 * (R15.loc[SEL15, col] / R15.loc[SEL15, a]- 1) for col, a in ant.items() if col in R15.columns}).round(1))

### 15.7 Controles (asserts) y autotest sintético

Se verifica: $p_0$ reproduce las columnas de `d`; $\rho_{mor}$ implícito $\in(0,1)$; $N_v+N_m=N_t$ y $share\in(0{,}45;0{,}55)$; $S_s(r_s)\in(0,1)$ y $S_{Total}(65)$ reproduce `S65`; $TBP_C^{varones}+TBP_C^{mujeres}=TBP_C^{total}$ (también calculado por fuera); con $share=1$ varón, $r=65$ y $\bar A$ constante el modelo **reproduce el $TBP_C$ base**; linealidad exacta en $\bar A$ y $\tau$; más $\rho$ ⇒ menos $TBP_C$; $N_{efectivo}S=\text{Pob}_{ONU}$; $TBP_A=D^*TBP_B$. Si la sección 10 está en memoria, se contrasta además el $J$ reponderado propio contra el suyo.

In [ ]:
MSGS15 = controles_modelo_avanzado(RES15, d)
if HAY_SEXO15 and "JRW" in globals():                    # contraste con la sección 10 (J reponderado total, escenario persistencia)
    assert np.allclose(RES15["JJ_rw"]["J65"].values, JRW["persistencia"].values, rtol=1e-9), "J reponderado propio != sección 10"
    print("OK: J (65+ total) reponderado propio = JRW['persistencia'] de la sección 10")
autotest_modelo_avanzado()

### 15.8 Salida: `tbp_modelo_avanzado_1950_2035.csv`

Una fila por cohorte: pasos 0–5 de $TBP_C$, componentes (share, $S_s$, $E_{r,s}$, $\bar A_s$, $J^*$, $m$), $TBP_C/TBP_A/TBP_B$ por sexo, escenarios de $\rho$ y de natalidad, y la sensibilidad "todos a 65" (`*_m65`). Se escribe en `../datos/procesados/` si existe; si no, en `OUT`. Si hubo EPH real también se guarda `abar_eph_por_sexo_cohorte.csv` (Ā por sexo, derivado de la EPH real).

In [ ]:
SAL15 = R15.copy()
SAL15.insert(0, "nota", "niveles condicionados a supuestos; no es predicción. Ā: EPH asalariados (cota inferior). ρ post-2023 y natalidad >=2025: supuestos/escenarios")
os.makedirs(DEST15, exist_ok=True)
f15 = os.path.join(DEST15, "tbp_modelo_avanzado_1950_2035.csv")
SAL15.round(6).reset_index().to_csv(f15, index=False)
assert len(SAL15) == 86 and SAL15.drop(columns=["nota"]).select_dtypes("number").notna().all().all()
print("Escrito:", os.path.abspath(f15), SAL15.shape)
print("Gráficos en", OUT, ": seccion_15_pasos_TBP_C.png, seccion_15_sexo_retiro.png, seccion_15_rho_moratorias.png, seccion_15_migracion.png")

### 15.9 Lectura y límites

* **Qué se puede decir.** (1) Pasar de un bloque único a *varones 65 / mujeres 60* **baja** $TBP_C$ en todas las cohortes (la mayor parte por el denominador $J^*$: hay más pasivas de 60 a 64) aun con más sobrevivientes de las mujeres a los 60 y menos años de aporte; (2) el nivel depende mucho más de $\bar A$ y de $\rho$ que de los demás refinamientos; (3) un $\rho$ más alto (menos moratorias) **reduce** $TBP_C$ de forma importante: es uno de los cambios más grandes de la sección (−17 % con 'mitad', −28 % con 'sin moratorias') y es un **supuesto**; (4) el ajuste de migración mueve $TBP_C$ unos ±2–7 % en la mayoría de las cohortes ≤2024 y +22 % en la cohorte 2024, donde captura sobre todo el desfase DEIS/ONU; en cohortes ≥2025 es ≈0 por construcción.
* **Qué no se puede decir.** Que alguno de los pasos sea "el verdadero" ni que los escenarios de moratorias o natalidad sean pronósticos. No hay significancia estadística: son cuentas condicionales, no estimaciones con error muestral.
* **Límites.** $\bar A$ por sexo: EPH urbana, sólo asalariados con descuento (cota inferior), pseudo-panel con efecto de cohorte aditivo; Ā de cohortes ≥2025 y de las más jóvenes es **extrapolación**. $\rho$ y $\tau$ iguales para ambos sexos y tomados en $t+65$; $\tau$ constante (supuesto). $J^*$ es un stock en $t+65$ (las mujeres se retiran 5 años antes). $\rho_{mor}$ implícito sale de datos digitalizados (aprox.) y supone igual denominador. En el paso 4, $J^*$ de las cohortes >2030 incluye mujeres 60–64 de cohortes sin escenario (ONU). Migración: el factor $m$ **no** es migración pura.
* **No verificado.** No se contrastó $\rho_{mor}$ con ninguna cifra oficial publicada de haber medio por tipo de beneficio; el reparto de $\rho$ y $\tau$ por sexo no existe en las fuentes cargadas.